# Load Customer Churn CSV into Snowflake

This notebook reads `customer_churn_training.csv.gz` into a pandas DataFrame and saves it as `CUSTOMER_CHURN_DB.CHURN.CUSTOMER_CHURN_TRAINING`.

It is safe to re-run: the database and schema are recreated each time.

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd

session = get_active_session()
session.use_role("ACCOUNTADMIN")
session.use_warehouse("COMPUTE_WH")

print(session.sql("SELECT CURRENT_ROLE(), CURRENT_WAREHOUSE()").collect()[0])

In [ ]:
%%sql -r create_db_schema_result
CREATE OR REPLACE DATABASE CUSTOMER_CHURN_DB;
CREATE OR REPLACE SCHEMA CUSTOMER_CHURN_DB.CHURN;

In [ ]:
churn_df = pd.read_csv(
    "customer_churn_training.csv.gz",
    compression="gzip",
    parse_dates=["SNAPSHOT_DATE"],
)

# Count features are whole numbers; use nullable Int64 to keep the NULLs
int_cols = [
    "LOGINS_LAST_12_MONTHS", "LOGINS_LAST_24_MONTHS",
    "TRANSACTIONS_LAST_12_MONTHS", "TRANSACTIONS_LAST_24_MONTHS",
    "CUSTOMER_AGE",
]
churn_df[int_cols] = churn_df[int_cols].astype("Int64")

print(churn_df.shape)
print(churn_df.dtypes)
churn_df.head()

In [ ]:
session.write_pandas(
    churn_df,
    table_name="CUSTOMER_CHURN_TRAINING",
    database="CUSTOMER_CHURN_DB",
    schema="CHURN",
    auto_create_table=True,
    overwrite=True,
    use_logical_type=True,
)
print("Saved to CUSTOMER_CHURN_DB.CHURN.CUSTOMER_CHURN_TRAINING")

In [ ]:
%%sql -r churn_summary
SELECT
    COUNT(*)                    AS total_rows,
    COUNT_IF(CHURN = 1)         AS churned,
    COUNT_IF(CHURN = 0)         AS retained,
    MIN(SNAPSHOT_DATE)          AS min_snapshot,
    MAX(SNAPSHOT_DATE)          AS max_snapshot
FROM CUSTOMER_CHURN_DB.CHURN.CUSTOMER_CHURN_TRAINING;